# Gen Python API — Walkthrough

This notebook demonstrates the core capabilities of the `gen` Python package:
initializing a repository, importing a reference sequence, applying variants
from a VCF, searching for sequence motifs, and visualizing the resulting graph.

In [ ]:
import os
import tempfile

import gen

print(f"gen version: {gen.__version__}")

## Initialize a repository

A `Repository` is the top-level object. Passing a path to a `.gen` directory
creates a new repository there; omitting the path searches the current working
directory for an existing one.

In [ ]:
tmpdir = tempfile.mkdtemp()
repo = gen.Repository(os.path.join(tmpdir, "gen"))
print(f"Repository created at: {repo.gen_dir}")

## Import a reference sequence

`import_fasta` loads a FASTA file and creates a sequence graph for each
contig. The `name` argument sets the collection and `sample` identifies the
sample (defaults to `"default"` when omitted).

In [ ]:
fasta_path = os.path.join(tmpdir, "reference.fa")
with open(fasta_path, "w") as f:
    f.write(">m123\nATCGATCGATCGATCGATCGGGAACACACAGAGA\n")

result = repo.import_fasta(fasta_path, name="demo", sample="reference")
print(result)

## Explore block groups

Each contig in the imported FASTA becomes a *block group* — the internal
representation of a sequence graph.

In [ ]:
bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) after import:")
for bg in bgs:
    print(f"  name={bg.name!r}  collection={bg.collection_name!r}  sample={bg.sample_name!r}")

## Apply variants from a VCF

`update_with_vcf` reads a VCF file and creates a new sample whose graph
incorporates the specified variants. The `sample` argument selects which
sample column to read from the VCF.

In [ ]:
vcf_path = os.path.join(tmpdir, "variants.vcf")
with open(vcf_path, "w") as f:
    f.write(
        "##fileformat=VCFv4.1\n"
        "##contig=<ID=m123,length=34>\n"
        "##FORMAT=<ID=GT,Number=1,Type=String,Description=\"Genotype\">\n"
        "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tsample1\n"
        "m123\t3\t.\tCG\tC\t.\t.\t.\tGT\t1/1\n"
        "m123\t10\t.\tT\tTAGA\t.\t.\t.\tGT\t1/1\n"
    )

result = repo.update_with_vcf(vcf_path, name="demo", sample="sample1", parent_samples=["reference"])
print(result)

In [ ]:
bgs = repo.get_block_groups()
print(f"{len(bgs)} block group(s) after VCF update:")
for bg in bgs:
    print(f"  name={bg.name!r}  collection={bg.collection_name!r}  sample={bg.sample_name!r}")

## Search for a sequence motif

`repo.search(query)` finds all occurrences of a sequence across every block
group. Results are `(BlockGroup, [GraphLocus])` pairs; each `GraphLocus`
describes where in the graph the match lives.

In [ ]:
query = "ATCGATCG"
results = repo.search(query)

print(f"Searching for {query!r}:")
for bg, loci in results:
    if loci:
        print(f"  {bg.name!r} (sample={bg.sample_name!r}): {len(loci)} match(es)")
        for locus in loci:
            print(f"    strand={locus.strand}  blocks={len(locus.blocks)}")

## Visualize the graph

`bg.plot()` returns an interactive `GenGraphWidget`. This requires the Jupyter
extra (`pip install gen[jupyter]`). In a notebook environment it renders an
interactive terminal-style graph viewer you can navigate with the mouse.

In [ ]:
reference_bg = next(bg for bg in bgs if bg.sample_name == "reference")

if gen.GenGraphWidget is not None:
    widget = reference_bg.plot(cols=120)
    display(widget)
else:
    print("Jupyter widget not available. Install with: pip install gen[jupyter]")